# 03 - Inference
The saved artefact `models/heart_model.joblib` is a single sklearn `Pipeline`: feature engineering, imputation, scaling and encoding, then the model. Raw patient JSON goes in directly.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Image, display
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

In [2]:
from heart.predict import load_model, predict_records
model = load_model()
model

Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('clinical', ClinicalFeatures()),
                                 ('columns',
                                  ColumnTransformer(transformers=[('num',
                                                                   Pipeline(steps=[('impute',
                                                                                    SimpleImputer(strategy='median')),
                                                                                   ('scale',
                                                                                    StandardScaler())]),
                                                                   ['age',
                                                                    'trestbps',
                                                                    'chol',
                                                                    'thalach',
                                                                    'oldpeak',
                                                                    'ca',
                                                                    'hr_reserve_pct',
                                                                    'chol_age_ratio']),
                                                                  ('bin',
                                                                   Pipeline(steps=[('impute',
                                                                                    SimpleImputer(...
                                                                   ['sex',
                                                                    'fbs',
                                                                    'exang']),
                                                                  ('cat',
                                                                   Pipeline(steps=[('impute',
                                                                                    SimpleImputer(strategy='most_frequent')),
                                                                                   ('onehot',
                                                                                    OneHotEncoder(handle_unknown='ignore',
                                                                                                  sparse_output=False))]),
                                                                   ['cp',
                                                                    'restecg',
                                                                    'slope',
                                                                    'thal'])],
                                                    verbose_feature_names_out=False))])),
                ('model',
                 LogisticRegression(C=0.5, class_weight='balanced',
                                    max_iter=2000, random_state=42,
                                    solver='liblinear'))])

In [3]:
patient = json.loads((ROOT/'sample_request.json').read_text())
predict_records([patient])

[{'prediction': 1,
  'label': 'disease',
  'probability_disease': 0.8945,
  'confidence': 0.8945,
  'risk_band': 'high'}]

In [4]:
# missing optional fields are imputed by the pipeline
predict_records([{**patient, 'ca': None, 'thal': None}])

[{'prediction': 1,
  'label': 'disease',
  'probability_disease': 0.6799,
  'confidence': 0.6799,
  'risk_band': 'high'}]

In [5]:
from heart.data import load_clean
df = load_clean().sample(8, random_state=1)
out = pd.DataFrame(predict_records(df.drop(columns='target').to_dict('records')))
out.insert(0, 'actual', df['target'].values); out

,actual,prediction,label,probability_disease,confidence,risk_band
0,0,0,no_disease,0.4134,0.5866,moderate
1,0,0,no_disease,0.4774,0.5226,moderate
2,0,0,no_disease,0.2220,0.7780,low
3,1,1,disease,0.9745,0.9745,high
4,1,0,no_disease,0.1879,0.8121,low
5,0,0,no_disease,0.0555,0.9445,low
6,0,0,no_disease,0.0680,0.9320,low
7,1,1,disease,0.9699,0.9699,high


## Same model through the REST API
```bash
curl -X POST http://localhost:8088/predict -H 'content-type: application/json' -d @sample_request.json
```

In [6]:
import urllib.request
try:
    req = urllib.request.Request('http://localhost:8088/predict', json.dumps(patient).encode(), {'content-type':'application/json'})
    print(urllib.request.urlopen(req, timeout=3).read().decode())
except Exception as e:
    print('API not running locally:', e)

{"prediction":1,"label":"disease","probability_disease":0.8945,"confidence":0.8945,"risk_band":"high","model_name":"logistic_regression","model_version":"1.0.0"}
